# TETOUAN-POWER-ML
## 1. Problema
Predecir Zona 1 a 60 minutos con información causal.

In [1]:
from pathlib import Path
import os, subprocess, sys
if not Path('pyproject.toml').exists():
    if not Path('ML').exists():
        subprocess.run(['git','clone','--branch','feature/tetouan-power-ml','https://github.com/albarracinelvys-droid/ML.git'],check=True)
    os.chdir('ML')
subprocess.run([sys.executable,'-m','pip','install','-e','.','xgboost','lightgbm','catboost','optuna'],check=True)

CompletedProcess(args=['C:\\Users\\gomez\\OneDrive\\Escritorio\\ml\\project\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-e', '.', 'xgboost', 'lightgbm', 'catboost', 'optuna'], returncode=0)

## 2. Dataset
UCI 849, CC BY 4.0. No modificar el archivo original.

In [2]:
from tetouan.data import download, read
path=download('data/raw/tetouan.csv')
frame=read(path)
print(frame.shape, frame.index.min(), frame.index.max())

## 3. EDA e ingeniería
Exploración inicial solo del tramo de entrenamiento.

In [3]:
display(frame.iloc[:int(len(frame)*.7)].describe())

,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
count,36691.000000,36691.000000,36691.000000,36691.000000,36691.000000,36691.000000,36691.000000,36691.000000
mean,19.239295,67.656150,1.998043,207.659901,86.544220,32956.889155,20444.842907,20024.412285
std,6.181506,16.082587,2.357462,285.726940,135.881914,7255.594859,5067.724969,6464.384037
min,3.247000,11.340000,0.050000,0.004000,0.019000,13895.696200,8560.081466,8189.908069
25%,14.340000,57.380000,0.077000,0.066000,0.126000,26703.423335,16298.757335,15290.181820
50%,19.160000,69.520000,0.086000,18.210000,16.440000,33089.361700,20444.350580,18071.630770
75%,24.040000,81.400000,4.913000,383.200000,119.800000,38195.880400,23875.623860,24821.567400
max,40.010000,93.600000,4.978000,1163.000000,936.000000,52204.395120,37408.860760,47598.326360


## 4. Modelado clásico y RealMLP
Presupuesto inicial de 16 épocas, CPU.

In [4]:
from tetouan.train import run
run(path,'artifacts',epochs=16)

## 5. Evaluación
Selección en validación; test no interviene en selección.

In [5]:
import pandas as pd
display(pd.read_csv('artifacts/validation.csv'))
display(pd.read_csv('artifacts/test.csv'))

,model,train_seconds,mae,rmse,r2
0,realmlp,33.364042,659.221948,880.553527,0.982176
1,lgbm,0.435426,659.926360,913.241408,0.980828
2,hist,1.383831,672.326636,928.163932,0.980197
3,linear,0.078380,662.857798,937.880103,0.979780
4,ridge,0.056532,665.209348,939.541116,0.979708
5,catboost,1.468760,725.024415,982.101502,0.977828
6,xgb,0.624156,812.219902,1101.240221,0.972123
7,forest,15.367713,753.576419,1129.533684,0.970672
8,dummy,0.062731,5471.673315,6600.092297,-0.001358


,model,inference_seconds,mae,rmse,r2
0,dummy,0.001965,6091.891877,7229.013572,-0.420259
1,linear,0.005829,784.240785,991.864412,0.973263
2,ridge,0.004050,790.020699,996.760798,0.972998
3,forest,0.052373,1531.583531,2034.823734,0.887471
4,hist,0.022534,1005.755841,1276.831354,0.955693
5,realmlp,0.113413,588.629300,798.929320,0.982653
6,xgb,0.011396,1161.099647,1471.353384,0.941164
7,lgbm,0.022385,1010.732518,1273.894580,0.955896
8,catboost,0.005211,876.105540,1148.181360,0.964171
9,persistence,NaN,1821.410402,2504.176870,0.829573


## 6. Interpretabilidad
Importancias por permutación; asociaciones, no efectos causales.

In [6]:
display(pd.read_csv('artifacts/importance.csv'))

,Unnamed: 0,0
0,Humidity,-4.380489
1,diffuse flows,-2.178078
2,general diffuse flows,-1.378993
3,Temperature,-0.587757
4,lag_12,-0.404655
5,std_144,-0.329116
6,mean_36,-0.018928
7,Wind Speed,0.207364
8,year_sin,0.543196
9,Zone 3 Power Consumption,0.982999


## 7. Mini-despliegue y comunicación
Ejecutar localmente `python -m streamlit run app.py`. En Colab se valida inferencia sin exponer un servidor público.

In [7]:
import joblib
from tetouan.inference import predict
print(predict(joblib.load('artifacts/model.joblib'),frame))